# 第 26 课｜数据来源大全

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`26_youtube-Ry5Ab7zZ9mw/index.html`  
> **所属模块**：P1-M6 · 数据契约与策略报告

## 本节量化问题

不同数据源字段不一致时，怎样建立稳定的研究输入？

## 准确定义与策略报告位置

数据适配层把供应商字段映射到项目 schema，并记录来源、获取时间、权限、复权和字段语义。

- **核心关系**：数据契约不是公式，而是 source、as_of、timezone、frequency、adjustment、schema 和许可的共同约束。
- **写入报告**：数据来源、许可、schema 和版本
- **P2 衔接**：P2 第 43–46 课：数据接口、SQLite 和可复现流水线。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 把三种供应商字段映射到统一 schema

示例是离线构造的供应商风格数据，不是实时接口结果。适配后统一为 `date/symbol/open/high/low/close/volume`。

In [2]:
yfinance_style = pd.DataFrame({
    "Date": ["2025-01-02"], "Ticker": ["SIM_A"], "Open": [10.0],
    "High": [10.5], "Low": [9.9], "Close": [10.3], "Volume": [100_000],
})
akshare_style = pd.DataFrame({
    "日期": ["2025-01-02"], "代码": ["SIM_A"], "开盘": [10.0],
    "最高": [10.5], "最低": [9.9], "收盘": [10.3], "成交量": [100_000],
})
tushare_style = pd.DataFrame({
    "trade_date": ["20250102"], "ts_code": ["SIM_A"], "open": [10.0],
    "high": [10.5], "low": [9.9], "close": [10.3], "vol": [100_000],
})

def normalize_frame(frame, mapping, source):
    normalized = frame.rename(columns=mapping).copy()
    normalized["date"] = pd.to_datetime(normalized["date"])
    normalized["source"] = source
    return normalized[["date", "symbol", "open", "high", "low", "close", "volume", "source"]]

frames = [
    normalize_frame(yfinance_style, {"Date":"date","Ticker":"symbol","Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"}, "yfinance_style"),
    normalize_frame(akshare_style, {"日期":"date","代码":"symbol","开盘":"open","最高":"high","最低":"low","收盘":"close","成交量":"volume"}, "akshare_style"),
    normalize_frame(tushare_style, {"trade_date":"date","ts_code":"symbol","vol":"volume"}, "tushare_style"),
]
display(pd.concat(frames, ignore_index=True))

,date,symbol,open,high,low,close,volume,source
0,2025-01-02,SIM_A,10.0,10.5,9.9,10.3,100000,yfinance_style
1,2025-01-02,SIM_A,10.0,10.5,9.9,10.3,100000,akshare_style
2,2025-01-02,SIM_A,10.0,10.5,9.9,10.3,100000,tushare_style


## 方法边界与阶段验收

免费接口可能限流、改版或回溯修订；研究结果必须能追溯到具体快照。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
